### Practice: Large Language Models and Their Implications
<!-- ![img](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fbucketeer-e05bbc84-baa3-437e-9518-adb32be77984.s3.amazonaws.com%2Fpublic%2Fimages%2F4470ce74-e595-4750-92a5-5f21f040df6d_577x432.jpeg) -->
![img](https://i.imgur.com/QGYa2J8.jpeg)

In this notebook, you're gonna play with some of the largest language models on the Internet.

_Based on works of: Tim Dettmers, Ruslan Svirschevsky, Artem Chumachenko, Younes Belkada, Felix Marty, Yulian Gilyazev, Gosha Zolotov, Andrey Ishutin,  Elena Volf, Artemiy Vishnyakov, Svetlana Shirokovskih, Maria Ivanova.

### Part 1: prompt engineering (4 points total)

Your task is to prompt-engineer a **base** language model into solving a few tasks for you. A base model only *continues text*, it was never trained to help you, so you steer it with careful prompts and few-shot examples.

__Which model?__ run a base model locally, for example [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base).

__Deliverable:__ for each of the 4 problems, show your **prompt(s)**, the **code** you ran, the **model's output**, and a short **description** of your solution.

__How a base model behaves:__ it continues whatever you write. Begin a dialogue and it continues the dialogue;
supply a few worked examples and it imitates their format. Load the model in the two cells below, then use the
`generate()` helper in your solutions.

__It is fine to re-run and tweak.__ Base models are sensitive to wording: if the output is off, adjust the prompt
or add a few-shot example. If you need many retries for one task, your prompt probably needs rethinking.

In [32]:
%pip install -q "transformers>=5.0" "accelerate>=1.0"
import torch, transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm, trange

# device = 'cuda'
# major, _ = torch.cuda.get_device_capability()
# dtype = torch.bfloat16 if major >= 8 else torch.float16   # T4 (compute 7.5) has no native bf16 -> use fp16
# print("transformers", transformers.__version__, "| GPU:", torch.cuda.get_device_name(), "| dtype:", dtype)

device = 'mps'
dtype = torch.bfloat16
from termcolor import colored

Note: you may need to restart the kernel to use updated packages.


In [33]:
MODEL_NAME = "Qwen/Qwen3-4B-Base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=dtype).to(device).eval()


def generate(prompt, max_new_tokens=64, **kwargs):
    """Continue `prompt` with the base model; return only the newly generated text.
    Pass do_sample=True (+ temperature/top_p/top_k) for creative output, or nothing for greedy."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)


# smoke test: a base model just continues the text
print(colored(generate("Once upon a time, deep in the forest,", max_new_tokens=40,
               do_sample=True, temperature=0.8, top_p=0.9), color = 'blue'))

Could not reach the Hub ([Errno 8] nodename nor servname provided, or not known). Using cached commit hash for 'Qwen/Qwen3-4B-Base'.
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Base/tree/main/additional_chat_templates
Retrying in 1s [Retry 1/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Base/tree/main/additional_chat_templates
Retrying in 2s [Retry 2/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Base/tree/main/additional_chat_templates
Retrying in 4s [Retry 3/5].
'[Errno 8] nodename nor servname provided, or not known' thrown while requesting GET https://huggingface.co/api/models/Qwen/Qwen3-4B-Base/tree/main/additional_chat_templates
Retrying in 8s [Retry 4/5].
'[Errno 8] nodename nor servname provided, or not known' th

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

 lived a young elf named Lir. Lir had always been curious about the world beyond the forest. He often listened to the stories of his elders about the lands far away, but he longed


__Task 1 (1 pt):__ arange a conversation between any two of the following:

- a celebrity or politician of your choice
- any fictional character (except Darth Vader)
- yourself

Compare two setups: a) you prompt with character names only b) you supply additional information (see example).

 __Example:__ Tony is talking to Darth Vader. Black text is written manually, blue text is generated.

![img](https://i.imgur.com/a1QhKF7.png)

In [34]:
short_prompt = """Taylor Swift is talking to Tony Soprano. 

Tony:"""

### they use greedy sampling by default...
print(short_prompt, 
      colored(generate(short_prompt, max_new_tokens=256), 'light_blue'))

Taylor Swift is talking to Tony Soprano. 

Tony:  "Taylor, I've been thinking about you. You're a talented artist, but you need to be more... *cough* *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*... *cough*


In [35]:
short_prompt = """Taylor Swift is talking to Tony Soprano. 

Tony:"""

print(short_prompt, 
      colored(generate(short_prompt, max_new_tokens=256, 
                       do_sample=True, temperature=0.8, top_p=0.9), 'light_blue'))

Taylor Swift is talking to Tony Soprano. 

Tony:  So you're the new girl on the block, huh? 

Taylor: Yeah, I guess so. 

Tony: What brings you to New Jersey, anyway? 

Taylor: I'm just passing through. 

Tony: Oh, I see. Well, you're in the right place. This town has a lot to offer. 

Taylor: I'm not sure I'm looking for much, to be honest. 

Tony: Well, I can help you find some entertainment. You know, we have some pretty good nightclubs and bars around here. 

Taylor: I don't really do bars or clubs. I'm more of a solo traveler. 

Tony: Oh, okay. Well, I can introduce you to some people who might be more your speed. 

Taylor: I don't really need anyone's help. I can take care of myself. 

Tony: That's good to hear. But if you ever need anything, just let me know. I'm always happy to help. 

Taylor: I appreciate that. 

Tony: So, what do you do for a living? 

Taylor: I'm a writer. 

Tony: Ah, I see. Well, writing can be tough. But you have to keep pushing forward. 

Taylor:


In [36]:
long_prompt = """Taylor Swift is talking to Tony Soprano. Taylor Swift is popular singer, 
who gives shows all around the world and has a lot of fans. Tony Soprano is New Jersey mob leader, 
who likes to eat gabagol, visit mistresses, shot peoples' knees and feeding ducks.

Tony:"""

print(long_prompt, 
      colored(generate(long_prompt, max_new_tokens=256, 
                       do_sample=True, temperature=0.8, top_p=0.9), 'light_blue'))

Taylor Swift is talking to Tony Soprano. Taylor Swift is popular singer, 
who gives shows all around the world and has a lot of fans. Tony Soprano is New Jersey mob leader, 
who likes to eat gabagol, visit mistresses, shot peoples' knees and feeding ducks.

Tony:  Hey, what's up?
Taylor: Hi, Tony. How's it going?
Tony: Good, you?
Taylor: Fine, just enjoying my show.
Tony: I heard about your show, it's really good.
Taylor: Thank you, Tony. What about you?
Tony: I'm doing well, just finishing up a few things.
Taylor: So, what's next for you?
Tony: Well, I'm looking into some new business opportunities.
Taylor: Oh, that sounds interesting. What kind of business?
Tony: I'm thinking of starting a restaurant.
Taylor: A restaurant? That's a great idea. What kind of food do you want to serve?
Tony: I was thinking of serving Italian food.
Taylor: Italian food is great, I love Italian food.
Tony: Yeah, it's my favorite too. Do you want to come over to my restaurant sometime?
Taylor: I'd love to,

__Please choose task 2a or 2b (1pt)__ depending on your model (you can do both, but you will be awarded points for one of these two tasks).

__Task 2a:__ zero-shot translation. Take the first verse of [Edgar Allan Poe's "Raven"](https://www.poetryfoundation.org/poems/48860/the-raven) and __translate it into French.__ (You are free to use any other text of at least the same size)

Original text:

```
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
```

Verify your translation by converting french back into english using a public machine translation service.

__Task 2b:__ toxicity classification for [SetFit/toxic_conversations](https://huggingface.co/datasets/SetFit/toxic_conversations).

Make the model solve binary classification (toxic vs not toxic) in the few shot mode. For few-shot examples, use 2-3 toxic and 2-3 non-toxic non-toxic examples. Measure accuracy on at least 25 samples. You may need to try several different prompts before you find the one that works.

In [53]:
poem = """
Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Only this and nothing more.”
"""

one_shot_eng = """

Let me tell you about my marvelous god, how he hides in the hexagons
of the bees, how the drought that wrings its leather hands
above the world is of his making, as well as the rain in the quiet minutes
that leave only thoughts of rain.
"""
# An atom is working and working, an atom is working in deepest
# night, then bursting like the farthest star; it is far
# smaller than a pinprick, far smaller than a zero and it has no
# will, no will toward us.
# This is why the heart has paced and paced,
# will pace and pace across the field where yarrow
# was and now is dust. A leaf catches
# in a bone. The burrow’s shut by a tumbled clod
# and the roots, upturned, are hot to the touch.
# How my god is a feathered and whirling thing; you will singe your arm
# when you pluck him from the air,
# when you pluck him from that sky
# where grieving swirls, and you will burn again
# throwing him back.
# """

one_shot_fr = """
Laissez-moi vous parler de mon dieu merveilleux, de la façon dont il se cache dans les hexagones
des abeilles, dont la sécheresse qui tord ses mains de cuir
au-dessus du monde est son œuvre, tout comme la pluie, dans ces instants de calme
qui ne laissent place qu’à des pensées de pluie.
"""
# Un atome est à l’œuvre, inlassablement, un atome travaille au cœur de la nuit
# la plus profonde, avant d’éclater telle l’étoile la plus lointaine ; il est bien
# plus petit qu’une piqûre d’épingle, bien plus petit qu’un zéro, et il n’a aucune
# volonté, aucune volonté à notre égard.
# C’est pourquoi le cœur a arpenté, inlassablement,
# et continuera d’arpenter le champ où se dressait autrefois l’achillée,
# réduite aujourd’hui en poussière. Une feuille se prend
# dans un os. L’entrée du terrier est obstruée par une motte de terre effondrée,
# et les racines, mises à nu, sont brûlantes au toucher.
# Mon dieu est une créature emplumée qui tourbillonne ; vous vous brûlerez le bras
# en le saisissant dans les airs,
# en le tirant de ce ciel
# où tournoie le deuil, et vous vous brûlerez à nouveau
# en le rejetant.
# """

prompt = f"""
Task: Translate the text below. Be precise. All phrases must match.
Source Language: English
Target Language: French

Text:
{one_shot_eng}

Translation:
{one_shot_fr}

Text:
{poem}

Translation:

"""

In [54]:
print(prompt)


Task: Translate the text below. Be precise. All phrases must match.
Source Language: English
Target Language: French

Text:


Let me tell you about my marvelous god, how he hides in the hexagons
of the bees, how the drought that wrings its leather hands
above the world is of his making, as well as the rain in the quiet minutes
that leave only thoughts of rain.


Translation:

Laissez-moi vous parler de mon dieu merveilleux, de la façon dont il se cache dans les hexagones
des abeilles, dont la sécheresse qui tord ses mains de cuir
au-dessus du monde est son œuvre, tout comme la pluie, dans ces instants de calme
qui ne laissent place qu’à des pensées de pluie.


Text:

Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Onl

In [76]:
from transformers import StoppingCriteria, StoppingCriteriaList

class StopOnTokens(StoppingCriteria):
    def __init__(self, stop_ids):
        self.stop_ids = stop_ids

    def __call__(self, input_ids, scores, **kwargs):
        return input_ids[0, -len(self.stop_ids):].tolist() == self.stop_ids

stop_ids = tokenizer.encode("\nText:", add_special_tokens=False)

print(prompt, 
      colored(generate(prompt, max_new_tokens=256, 
                       do_sample=True, temperature=0.8, top_p=0.95, 
                       stopping_criteria=StoppingCriteriaList([
        StopOnTokens(stop_ids)
    ])), 'light_blue')) 

## although domain is creative, large temperature hurted cohesion of answer
## model also generated new example, so i used stopping criteria


Task: Translate the text below. Be precise. All phrases must match.
Source Language: English
Target Language: French

Text:


Let me tell you about my marvelous god, how he hides in the hexagons
of the bees, how the drought that wrings its leather hands
above the world is of his making, as well as the rain in the quiet minutes
that leave only thoughts of rain.


Translation:

Laissez-moi vous parler de mon dieu merveilleux, de la façon dont il se cache dans les hexagones
des abeilles, dont la sécheresse qui tord ses mains de cuir
au-dessus du monde est son œuvre, tout comme la pluie, dans ces instants de calme
qui ne laissent place qu’à des pensées de pluie.


Text:

Once upon a midnight dreary, while I pondered, weak and weary,
Over many a quaint and curious volume of forgotten lore—
    While I nodded, nearly napping, suddenly there came a tapping,
As of some one gently rapping, rapping at my chamber door.
“’Tis some visitor,” I muttered, “tapping at my chamber door—
            Onl

Translation of answer:
```
Once, in the darkness of a grim night, while I pondered, weak and weary,
Over many curious and singular volumes of forgotten lore—
As I was nearly dozing off, suddenly, there came a tapping,
As if someone were gently knocking at my door.
“It is a visitor,” I said to myself, “tapping at my door—
Only that, and nothing more.”
```


__Task 3 (1pt):__ create a prompt and few-shot examples tha make the model __change the gender pronouns__ of the main actor in a given sentence in any direction of your choice. E.g. the doctor took off _his_ mask <-> the doctor took of _her_ mask.


In [124]:
from transformers import StopStringCriteria

gender_examples = """

Invert gender pronouns
her <-> his
she <-> he

source: the doctor took off his mask
target: the doctor took off her mask

source: he makes good sandwiches
target: she makes good sandwiches

source: she likes playing footbal
target: he likes playing footbal

source: she wears a black hat
target: he wears a black hat

"""

prompt = gender_examples + "\nsource: "

stopping_criteria = StoppingCriteriaList([StopStringCriteria(tokenizer, "\nsource:")])

for query in [
    "I use his pen to cut her paper",
    "She called him to help with her sister's homework",
]:

    print('source: ' + query + '\ntarget:', 
      colored(generate(prompt + query + '\ntarget:', max_new_tokens=256, 
                       do_sample=True, temperature=0.8, top_p=0.95,
                      stopping_criteria=stopping_criteria), 'light_blue')) 
    print(100*'-')


source: I use his pen to cut her paper
target:  I use her pen to cut his paper

source:
----------------------------------------------------------------------------------------------------
source: She called him to help with her sister's homework
target:  He called her to help with her sister's homework

source:
----------------------------------------------------------------------------------------------------


ooops it makes some mistakes...

__Task 4 (1pt):__ write a prompt and supply examples such that the model would __convert imperial units to metric units__ (miles -> kilometers; mph -> kph). More specifically, the model should rewrite a given sentence and replace all imperial units with their metric equivalents. After it works with basic distances and speed, try to find complicated examples where it does *not* work.

Please note that 1 mile is not equal to 1 km :)

In [126]:
from transformers import StopStringCriteria

metric_examples = """

Convert imperial units to metric units
1 mile = 1.6 km
2 miles = (2 * 1.6) km = 3.2 km
40 mph = (40 * 1.6) kmh = 64 kmh
80 mph = (80 * 1.6) kmh = 128 kmh

source: the hospital is 2.5 miles away
target: the hospital is 4 km away

source: it i snot allowed to drive faster than 30 mph here
target: it i snot allowed to drive faster than 48 kmh here
"""

prompt = metric_examples + "\nsource: "


for query in [
    "The school is 5 miles from home - it is a pretty long journey",
    "If I drive with constant speed of 50 mph I will get here in 80 minutes",
]:

    print('source: ' + query + '\ntarget:', 
      colored(generate(prompt + query + '\ntarget:', max_new_tokens=256, 
                       do_sample=True, temperature=0.8, top_p=0.95,
                      stopping_criteria=stopping_criteria), 'light_blue')) 
    print(100*'-')


source: The school is 5 miles from home - it is a pretty long journey
target:  The school is 8 km from home - it is a pretty long journey

source:
----------------------------------------------------------------------------------------------------
source: If I drive with constant speed of 50 mph I will get here in 80 minutes
target:  If I drive with constant speed of 80 kmh I will get here in 1 hour 40 minutes

source:
----------------------------------------------------------------------------------------------------


Although model successfully recovered that 5 maps to 8 (which wasn't shown in examples), it got fooled with time!

### Part 2: local inference & text-generation internals

In Part 1 we used `model.generate` as a black box. Now let's look **under the hood**: how a language model turns logits into text and implement a decoding strategy by hand.

We already loaded [`Qwen/Qwen3-4B-Base`](https://huggingface.co/Qwen/Qwen3-4B-Base) above: a dense 4B model that fits a free Colab T4 in fp16 (~8 GB).

## Text generation

**Comparison of strategies for language model text generation:**

| Strategy | Description | Pros & Cons |
| --- | --- | --- |
| Greedy Search | Chooses the word with the highest probability as the next word in the sequence. | **Pros:** Simple and fast. <br> **Cons:** Can lead to repetitive and incoherent text. |
| Sampling with Temperature | Introduces randomness in the word selection. A higher temperature leads to more randomness. | **Pros:** Allows exploration and diverse output. <br> **Cons:** Higher temperatures can lead to nonsensical outputs. |
| Nucleus Sampling (Top-p Sampling) | Selects the next word from a truncated vocabulary, the "nucleus" of words that have a cumulative probability exceeding a pre-specified threshold (p). | **Pros:** Balances diversity and quality. <br> **Cons:** Setting an optimal 'p' can be tricky. |
| Beam Search | Explores multiple hypotheses (sequences of words) at each step, and keeps the 'k' most likely, where 'k' is the beam width. | **Pros:** Produces more reliable results than greedy search. <br> **Cons:** Can lack diversity and lead to generic responses. |
| Top-k Sampling | Randomly selects the next word from the top 'k' words with the highest probabilities. | **Pros:** Introduces randomness, increasing output diversity. <br> **Cons:** Random selection can sometimes lead to less coherent outputs. |
| Length Normalization | Prevents the model from favoring shorter sequences by dividing the log probabilities by the sequence length raised to some power. | **Pros:** Makes longer and potentially more informative sequences more likely. <br> **Cons:** Tuning the normalization factor can be difficult. |
| Stochastic Beam Search | Introduces randomness into the selection process of the 'k' hypotheses in beam search. | **Pros:** Increases diversity in the generated text. <br> **Cons:** The trade-off between diversity and quality can be tricky to manage. |
| Decoding with Minimum Bayes Risk (MBR) | Chooses the hypothesis (out of many) that minimizes expected loss under a loss function. | **Pros:** Optimizes the output according to a specific loss function. <br> **Cons:** Computationally more complex and requires a good loss function. |

Documentation references:
- [reference for `AutoModelForCausalLM.generate()`](https://huggingface.co/docs/transformers/v4.29.1/en/main_classes/text_generation#transformers.GenerationMixin.generate)
- [reference for `AutoTokenizer.decode()`](https://huggingface.co/docs/transformers/main_classes/tokenizer#transformers.PreTrainedTokenizer.decode)
- Huggingface [docs on generation strategies](https://huggingface.co/docs/transformers/generation_strategies)

### Generation with HuggingFace

In [127]:
prompt = 'The first discovered martian lifeform looks like'
batch = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
print("Input batch (encoded):", batch)

output_tokens = model.generate(**batch, max_new_tokens=64, do_sample=True, temperature=0.8)
# greedy inference:                                        do_sample=False)
# beam search for highest probability:                     num_beams=4)

print("\nOutput:", tokenizer.decode(output_tokens[0].cpu()))

Input batch (encoded): {'input_ids': tensor([[  785,  1156, 11105, 48195,  1103,  2272,   627,  5868,  1075]],
       device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}

Output: The first discovered martian lifeform looks like a bacterium.
An international team of researchers has uncovered the first evidence of biological activity on another planet. A new study reports a strange yellowish rock with a greenish tint embedded in a layer of red dust off the planet’s surface. Microscopic inspection revealed that the rock is a bacterium, the researchers said.



#### Low-level code for text generation

In [128]:
prompt = "Moscow is the capital of"
# prompt = "Skippy, a young android, likes to dream about electric"

print(prompt, '\n')

voc = tokenizer.get_vocab()
voc_rev = {v:k for k, v in voc.items()}  # reverse vocab for decode

for i in range(10):
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    next_token_id = torch.multinomial(probs.flatten(), num_samples=1)

    next_token = tokenizer.decode(next_token_id)
    prompt += next_token

    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    top_tokens = sorted_indices[:5]
    print(f"Step #{i} candidates:")
    for t, p in zip (top_tokens, sorted_probs):
        t = voc_rev[t.item()]
        print(f"{t:<10}: {p:.4f} ")

    print(f'\nChosen token: {next_token}', end='\n\n', flush=True)

Moscow is the capital of 

Step #0 candidates:
ĠRussia   : 0.4648 
Ġthe      : 0.1514 
Ġwhich    : 0.1514 
Ġwhat     : 0.0491 
ĠRussian  : 0.0232 

Chosen token:  the

Step #1 candidates:
ĠRussian  : 0.7773 
Ġcountry  : 0.0498 
Ġworld    : 0.0183 
Ġformer   : 0.0143 
Ġlargest  : 0.0111 

Chosen token:  Russian

Step #2 candidates:
ĠFederation: 0.8711 
ĠEmpire   : 0.0337 
ĠRepublic : 0.0181 
Ġfederation: 0.0160 
ĠSoviet   : 0.0085 

Chosen token:  Federation

Step #3 candidates:
.         : 0.3066 
Ġand      : 0.2695 
,         : 0.2383 
.Ċ        : 0.0221 
Ġas       : 0.0195 

Chosen token: ,

Step #4 candidates:
Ġthe      : 0.3477 
Ġand      : 0.1641 
Ġa        : 0.0776 
Ġas       : 0.0603 
ĠRussia   : 0.0415 

Chosen token:  the

Step #5 candidates:
Ġlargest  : 0.7461 
Ġmost     : 0.1475 
ĠRussian  : 0.0226 
Ġcountry  : 0.0121 
Ġcentral  : 0.0073 

Chosen token:  largest

Step #6 candidates:
Ġcity     : 0.8359 
Ġcountry  : 0.0776 
Ġand      : 0.0471 
Ġin       : 0.0072 
Ġstate    : 0

**Task 5: write code for nucleus sampling generation (2 points)**:

Use the `nucleus_sampling()` template below. Look at the detailed generation code above for inspiration. __Please do not use model.generate__.

**Bonus task: write code for beam search (3 bonus points)**

I have implemented beam search with almost effective KV-caching. Every generation step is $O(\mathrm{seq\_len})$ but common prefix  is copied for each beam. Also, at each step KV-cache tensors are shuffled/reallocated, but in the end the cache is small for short input and generation is very fast.

In [240]:
from typing import Tuple, List

@torch.inference_mode()
def nucleus_sampling(model, tokenizer, prompt: str, prob: float = 0.5) -> Tuple[str, List[str]]:
    """generates the next token from the nucleus of tokens with cumulative probability up to param:prob"""

    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)
    logits = model.forward(**inputs).logits[0, -1, :]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    prob_cumsum = sorted_probs.cumsum(dim = -1)
    topk = torch.searchsorted(prob_cumsum, prob)

    next_token_pos = torch.multinomial(prob_cumsum[:topk+1].flatten(), num_samples=1).item()
    next_token_id = sorted_indices[next_token_pos]
    
    sampled_token = tokenizer.decode(next_token_id)
    possible_tokens = [tokenizer.decode(idx) for idx in sorted_indices[:topk+1]]

    # sampled_token should be a string token that was generated
    # possible_tokens should be a list of all tokens that have non-zero probability
    return sampled_token, possible_tokens

In [241]:
# Tests for nucleus sampling.
prompt = "Elbrus is the highest"

sampled, nucleus = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
print("sampled:", repr(sampled), "| nucleus size:", len(nucleus))

# 1) the sampled token must come from inside the nucleus
assert sampled in nucleus
assert len(nucleus) >= 1

# 2) the nucleus depends only on `prob`, not on the random draw -> it is deterministic across calls
_, nucleus_again = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
assert sorted(nucleus) == sorted(nucleus_again), "the nucleus itself should not be random"

# 3) a larger probability threshold never yields a smaller nucleus
_, small = nucleus_sampling(model, tokenizer, prompt, prob=0.1)
_, large = nucleus_sampling(model, tokenizer, prompt, prob=0.95)
assert len(large) >= len(small), "higher p should include at least as many tokens"

# 4) a tiny threshold collapses to the single most likely token
_, top1 = nucleus_sampling(model, tokenizer, prompt, prob=1e-6)
assert len(top1) == 1, "with p -> 0 the nucleus is just the top-1 token"

print("All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)")

sampled: ' mountain' | nucleus size: 3
All invariant checks passed. (Also eyeball that the nucleus tokens are sensible continuations.)


In [246]:
prompt = "Elbrus is the highest"
for i in range(10):

    sampled_token, possible_tokens = nucleus_sampling(model, tokenizer, prompt, prob=0.9)
    prompt += sampled_token

print(prompt)

Elbrus is the highest point in the Caucasus mountain range of Russia,


In [236]:
prompt = "Elbrus is the highest"
inputs = tokenizer(prompt, return_tensors='pt').to(device)
cache = model(**inputs, use_cache = True).past_key_values

inputs = tokenizer(prompt + ' mountain in the', return_tensors='pt').to(device)

cache.batch_repeat_interleave(4)

tokenizer.decode(model.generate(**inputs, 
                                past_key_values = cache, 
                                max_new_tokens = 10,      
                                num_beams = 4,
                                num_return_sequences=4,
                                ))

['Elbrus is the highest mountain in the Caucasus and the second highest in Europe, after',
 'Elbrus is the highest mountain in the Caucasus and the second highest in Europe after Mount',
 'Elbrus is the highest mountain in the Caucasus and the second highest in Europe. It',
 'Elbrus is the highest mountain in the Caucasus and the second highest in Europe after Mont']

In [831]:
from copy import deepcopy

class GeneratedSequence:

    """
    Class for stroging generated sequence
    and their cumulative logprobs
    """
    
    def __init__(self):
        self.tokens = []
        self.log_prob = 0.0

    def get_log_prob(self):

        return self.log_prob

    def __len__(self):

        return len(self.tokens)

    def __repr__(self):
        s = min(2, len(self))
        e = max(s, len(self) - 2) 
        sep = ' ... ' if s < e else ', '
        tokens_str = f'<{", ".join(map(str,self.tokens[:s]))}' + sep + f'{", ".join(map(str,self.tokens[e:]))}>'
        return f"GeneratedSequence {tokens_str}, log_prob = {self.log_prob:.4f}\n"

    def add_token(self, token, token_log_prob):

        """
        Modifies this sequence by adding new token
        """
        self.tokens.append(token)
        self.log_prob += token_log_prob

class SequenceStorage:

    """
    Class for stroging several instances of GeneratedSequence
    with useful utilities
    """

    def __init__(self, num_seq):

        self.sequence_array = [GeneratedSequence() for _ in range(num_seq)]

    def add_tokens(self, tokens, log_probs):
        
        """
        Add one token and its logprob to each sequence
        """

        for i, (token, logp) in enumerate(zip(tokens, log_probs)):
            sequence = self.sequence_array[i]
            sequence.add_token(token, logp)

    def __getitem__(self, i):

        return self.sequence_array[i]

    def __len__(self):

        return len(self.sequence_array)

    def __iter__(self):
        return iter(self.sequence_array)

    def __repr__(self):

        num_seq = len(self)
        desc = f"SequenceStorage with {num_seq} sequences\n"
        for i in range(num_seq):
            desc += f"\t{repr(self[i])}"

        return desc

    def append_sequence(self, sequence):

        """
        Append new sequence to storage
        Useful for store finished sequences after popping them 
        from active storage
        """

        self.sequence_array.append(sequence)

    def reorder(self, index):
        
        """
        replace array of sequences with it's shuffled version
        sequences are deep copied and can be safely duplicated
        """
        
        new_array = [deepcopy(self.sequence_array[i]) for i in index]
        self.sequence_array = new_array

def get_score(sequence, len_penalty):

    return sequence.get_log_prob() / len(sequence) ** len_penalty

def filter_storage(storage, score, len_penalty):

    keep = []
    for i in range(len(storage)):
        seq = storage[i]
        seq_score = get_score(seq, len_penalty)
        if seq_score >= score:
            keep.append(i)

    storage.reorder(keep)


def merge_storages(storage1, storage2, topk, len_penalty):

    scores1 = [get_score(seq, len_penalty) for seq in storage1]
    scores2 = [get_score(seq, len_penalty) for seq in storage2]
    
    all_scores = sorted(scores1 + scores2, reverse = True)
    score = all_scores[topk-1]

    filter_storage(storage1, score, len_penalty)
    filter_storage(storage2, score, len_penalty)

    return [seq for seq in storage1] + [seq for seq in storage2]


In [832]:
seq =  GeneratedSequence()
seq.add_token(1, -0.1)
seq.add_token(2, -0.2)
seq.add_token(3, -0.3)
seq.add_token(4, -0.1)
seq.add_token(5, -0.1)

seq

GeneratedSequence <1, 2 ... 4, 5>, log_prob = -0.8000

In [833]:
seq_storage = SequenceStorage(4)
seq_storage.add_tokens([1,2,3,4], [-0.5,-0.4, -0.3,-0.2])
print(seq_storage)

### change order
seq_storage.reorder([2,3,3,1])
print(seq_storage)

### each copy is independent
seq_storage[1].add_token(5, -0.1)
print(seq_storage)

SequenceStorage with 4 sequences
	GeneratedSequence <1, >, log_prob = -0.5000
	GeneratedSequence <2, >, log_prob = -0.4000
	GeneratedSequence <3, >, log_prob = -0.3000
	GeneratedSequence <4, >, log_prob = -0.2000

SequenceStorage with 4 sequences
	GeneratedSequence <3, >, log_prob = -0.3000
	GeneratedSequence <4, >, log_prob = -0.2000
	GeneratedSequence <4, >, log_prob = -0.2000
	GeneratedSequence <2, >, log_prob = -0.4000

SequenceStorage with 4 sequences
	GeneratedSequence <3, >, log_prob = -0.3000
	GeneratedSequence <4, 5, >, log_prob = -0.3000
	GeneratedSequence <4, >, log_prob = -0.2000
	GeneratedSequence <2, >, log_prob = -0.4000



In [475]:
from tqdm.auto import tqdm

@torch.inference_mode()
def beam_search(model, tokenizer, prompt: str, n_beams: int = 1, max_tokens = 64, len_penalty = 1.0) -> list[str]:
    
    """
    Generate k most probable trajectories

    Keep track of best n_beam active candidates 
    and at most best n_beam finished candidates (which end with EOS)

    This implementation is optimized for using KV cache for each generation
    It stores seperate cache for each active beam and reorders it at each step
    However, shared KV prefix is stored independently for each beam, wasting memory
    """    

    eos_idx = tokenizer.get_vocab()[tokenizer.eos_token]
    
    inputs = tokenizer(prompt, return_tensors='pt', return_token_type_ids=False).to(device)

    print("Starting PreFill...")
    output = model(**inputs, use_cache = True)
    print("PreFill Finished!")

    cache = output.past_key_values
    
    logits = output.logits[0, -1, :]
    log_probs = torch.log_softmax(logits, dim = -1)
    ### find top +1
    next_logprobs, next_tokens = torch.topk(log_probs, k = n_beams + 1, dim = -1, largest = True)

    active_candidates = SequenceStorage(n_beams)
    finished_candidates = SequenceStorage(0)

    ### if eos in topk, remove it from beams and add to finished
    if eos_idx in next_tokens:
        eos_logp = log_probs[next_tokens == eos_idx].item()
        active = torch.where(next_tokens != eos_idx)[0]
        
        candidate = GeneratedSequence()
        candidate.add_token(eos_idx, eos_logp)
        finished_candidates.append_sequence(candidate)
        
        next_logprobs = next_logprobs[active]
        next_tokens = next_tokens[active]

    ### otherwise, keep top n_beams 
    else:
        next_logprobs = next_logprobs[:-1]
        next_tokens = next_tokens[:-1]
    
    cum_log_probs = next_logprobs
    active_candidates.add_tokens(next_tokens, next_logprobs)
    
    seq_len = 1

    cache.batch_repeat_interleave(n_beams)

    bar = tqdm(total = max_tokens - 1, unit = 'token')

    while seq_len < max_tokens:

        output = model(input_ids = next_tokens.view(-1,1), past_key_values = cache)
        
        ### update cache for current beams
        cache = output.past_key_values

        ### get logprobs for next token at each beam [n_beam, vocab]
        logits = output.logits[:, -1, :]
        log_probs = torch.log_softmax(logits, dim = -1) # [n_beam, vocab]

        ### compute cumulative logprob for each candidate, i.e. beam + next_token
        all_log_probs = cum_log_probs.view(-1,1).expand(-1, log_probs.shape[-1]) + log_probs
        
        ### find global topk scores
        ### find 2*n_beams best scores
        ### because we have at most n_beams finished candidates,
        ### we will have at least n_beams active
        ### then find top n_beam active again
        flat_log_probs = all_log_probs.flatten()
        topk_scores, topk_indices = flat_log_probs.topk(2 * n_beams)
        
        beam_idx = topk_indices // log_probs.shape[-1]
        next_tokens = topk_indices % log_probs.shape[-1]

        # process finished
        active = next_tokens != eos_idx
        stop_idx = torch.searchsorted(active.cumsum(-1), n_beams)
        
        beam_idx = beam_idx[:stop_idx+1]
        next_tokens = next_tokens[:stop_idx+1]

        min_active_score = topk_scores[stop_idx] / (seq_len+1)**len_penalty

        ### now we are guaranteed to have exactly n_beams active candidates
        beam_idx_finished = beam_idx[next_tokens==eos_idx]
        beam_idx_active = beam_idx[next_tokens!=eos_idx]
        next_tokens = next_tokens[next_tokens!=eos_idx]
        next_log_probs = log_probs[beam_idx_active, next_tokens]

        # update cumulative log probs
        cum_log_probs = all_log_probs[beam_idx_active, next_tokens]
        
        ### fill new finished candidates
        for idx in beam_idx_finished:
            parent_beam = active_candidates[idx]
            sequence = deepcopy(parent_beam)
            sequence.add_token(eos_idx, log_probs[idx, eos_idx])
            finished_candidates.append_sequence(sequence)

        ### reorder candidates
        active_candidates.reorder(beam_idx_active)
        ### update candidates
        active_candidates.add_tokens(next_tokens, next_log_probs)

        ### reorder cache
        cache.reorder_cache(beam_idx_active)

        ### keep only finished candidates which are better than the worst active
        filter_storage(finished_candidates, min_active_score, len_penalty)

        seq_len += 1
        bar.update(1)

    bar.close()

    final_sequences = merge_storages(active_candidates, finished_candidates, n_beams, len_penalty)
    gen_answers, scores = [], []

    for seq in final_sequences:
        gen_answers.append(tokenizer.decode(seq.tokens))
        scores.append(get_score(seq, len_penalty).item())

    return gen_answers, scores

In [476]:
def memory_stats():
    gib = 1024 ** 3

    print(
        f"Tensor allocated: "
        f"{torch.mps.current_allocated_memory() / gib:.2f} GiB | "
        f"Driver allocated: "
        f"{torch.mps.driver_allocated_memory() / gib:.2f} GiB | "
        f"Recommended max: "
        f"{torch.mps.recommended_max_memory() / gib:.2f} GiB"
    )

In [477]:
import gc
gc.collect()
torch.mps.empty_cache()

In [478]:
memory_stats()

Tensor allocated: 8.75 GiB | Driver allocated: 10.98 GiB | Recommended max: 11.84 GiB


In [479]:
prompt = "Elbrus is the highest"
gen_answers, scores = beam_search(model, tokenizer, prompt, n_beams=4, max_tokens=256)

for answer, score in zip(gen_answers, scores):
    print(f"{score:.3f}", prompt,
          colored(answer, 'light_blue'))

Starting PreFill...
PreFill Finished!


  0%|          | 0/255 [00:00<?, ?token/s]

-0.283 Elbrus is the highest  mountain in Europe, located in the Caucasus Mountains of Russia. It stands at an impressive height of 5,642 meters (18,510 feet) above sea level. The mountain is a popular destination for mountaineers and adventurers from around the world, who come to test their skills and push their limits on one of the most challenging peaks in the world.
Elbrus is a dormant volcano, which means that it has not erupted for thousands of years. However, it is still considered to be an active volcano, as it has the potential to erupt in the future. The mountain is also home to a variety of plant and animal species, including rare and endangered species such as the Caucasian leopard and the Caucasian ibex.
There are two main summits on Elbrus, the North Peak and the South Peak. The North Peak is the higher of the two, standing at 5,642 meters (18,510 feet) above sea level, while the South Peak is slightly lower, standing at 5,621 meters (18,442 feet) above sea level. Both su

In [480]:
prompt = "Elbrus is the highest"
gen_answers, scores = beam_search(model, tokenizer, prompt, n_beams=20, max_tokens=20)

for answer, score in zip(gen_answers, scores):
    print(f"{score:.3f}", prompt,
          colored(answer, 'light_blue'))

Starting PreFill...
PreFill Finished!


  0%|          | 0/19 [00:00<?, ?token/s]

-0.342 Elbrus is the highest  mountain in Europe, standing at 5,642 meters (18,510
-0.357 Elbrus is the highest  peak in Europe, standing at 5,642 meters (18,510
-0.387 Elbrus is the highest  mountain in Europe, with an elevation of 5,642 meters (18,5
-0.387 Elbrus is the highest  peak in Europe, with an elevation of 5,642 meters (18,5
-0.402 Elbrus is the highest  mountain in Europe, standing at 5,642 metres (18,510
-0.428 Elbrus is the highest  peak in Europe, with an elevation of 18,510 feet (5,6
-0.447 Elbrus is the highest  mountain in Europe at 5,642 metres (18,510 ft)
-0.447 Elbrus is the highest  mountain in Europe at 5,642 metres (18,510 ft).
-0.453 Elbrus is the highest  mountain in Europe, with an elevation of 18,510 feet (5,6
-0.459 Elbrus is the highest  mountain in Europe, with an elevation of 18,510 feet above sea level.
-0.463 Elbrus is the highest  mountain in Europe at 5,642 m (18,510 ft)
-0.463 Elbrus is the highest  mountain in Europe at 5,642 m (18,510 ft).
-0.475 

In [482]:
### i dont know what is this
### but it translates from chinese as 'Pablo believes that a small extra task only takes a few hours."
prompt = """Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar."""
gen_answers, scores = beam_search(model, tokenizer, prompt, n_beams=20, max_tokens=20)

for answer, score in zip(gen_answers, scores):
    print(f"{score:.3f}", prompt,
          colored(answer, 'light_blue'))

Starting PreFill...
PreFill Finished!


  0%|          | 0/19 [00:00<?, ?token/s]

-0.512 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  翻译成中文
Pablo 认为一个小的额外任务只需要几个小时，
-0.523 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  翻译成中文

Pablo 认为一个小的额外任务只需要几个小时，
-0.535 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  翻译
Pablo 认为一个小的额外任务只需要几个小时，特别是考虑到
-0.543 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  翻译
Pablo认为一个小的额外任务可能需要几个小时，特别是考虑到他已经
-0.555 Pablo thought that small ex

In [488]:
### however, everything works fine for small beams
### note that Qwen overestimates the usefulness of code from HW2
prompt = """Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar."""
gen_answers, scores = beam_search(model, tokenizer, prompt, n_beams=4, max_tokens=20)

for answer, score in zip(gen_answers, scores):
    print(f"{score:.3f}", prompt,
          colored(answer, 'light_blue'))

Starting PreFill...
PreFill Finished!


  0%|          | 0/19 [00:00<?, ?token/s]

-1.117 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  
    Instead, he decided to use the code from HW2 and modify it. 
    He
-1.164 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  
    Instead, he decided to use the code from HW2. 
    He also decided to
-1.172 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He desided not to reinvent PagedAttention in ICL seminar.  
    Instead, he decided to use the code from HW2 and modify it. 
    The
-1.172 Pablo thought that small extra task will take a couple of hours, 
    especially provided that he has already done similar things in HW2.
    He de

### Part 3: Chain-of-thought prompting (4 points total)

![img](https://github.com/kojima-takeshi188/zero_shot_cot/raw/main/img/image_stepbystep.png)

---



In [489]:
import json
import random
import locale; locale.getpreferredencoding = lambda: "UTF-8"
!wget https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json -O aqua.json
data = list(map(json.loads, open("aqua.json")))

--2026-11-14 03:12:30--  https://raw.githubusercontent.com/kojima-takeshi188/zero_shot_cot/2824685e25809779dbd36900a69825068e9f51ef/dataset/AQuA/test.json
Распознаётся raw.githubusercontent.com (raw.githubusercontent.com)… 185.199.109.133, 185.199.110.133, 185.199.111.133, ...
Подключение к raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... соединение установлено.
HTTP-запрос отправлен. Ожидание ответа… 200 OK
Длина: 130192 (127K) [text/plain]
Сохранение в: «aqua.json»

aqua.json           100%[===================>] 127,14K   814KB/s    за 0,2s    

2026-11-14 03:12:31 (814 KB/s) - «aqua.json» сохранён [130192/130192]



In [490]:
print("Example:")
data[150]

Example:


{'question': 'Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?',
 'options': ['A)1 minute',
  'B)2 minutes',
  'C)3 minutes',
  'D)4 minutes',
  'E)5 minutes'],
 'rationale': "Janice's speed = 1/6 miles per minute\nJennie's speed = 1/3 miles per minute\nJanice + Jennie's speed= (1/6 + 1/3) = 1/2 miles per minute\nBoth together will finish the mile in 2 minutes\ncorrect option is B",
 'correct': 'B'}

### Naive solution

Here, we prompt the model to choose an answer to the example above (`data[150]`) out of the options given above. We're using a format that mimics grade school solution textbook.

Please note that there are minor formatting changes in options: an extra space and an opening bracket. Those may or may not be important :)

In [505]:
EXAMPLE_0SHOT = """
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
""".strip()

In [506]:
# solving an equation directly
batch = tokenizer(EXAMPLE_0SHOT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_0SHOT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))

[Prompt:]
Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Correct Answer:
[Generated:]  E
Answer Explanation: A: Janice's 10 miles per hour = 0.166667 miles per minute.
Jennie's 20 miles per hour = 0.333333 miles per minute.
0.166667 miles per minute + 0.333333 miles per minute = 0.50 miles per minute
1 ÷ 0.50 = 2 minutes.


And here's how you can solve this with few-shot chain-of-thought prompting.

You need to chang 3 things
- use a new field called **Rationale**, that contains a step-by-step solution to the problem
- add several few-shot examples of previously solved problems **with rationales**
- change the final prompt so that the model has to generate rationale before answering

In [507]:
EXAMPLE_3SHOT_CHAIN_OF_THOUGHT = """
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;\noriginal price = 100*1.6 = 160;\nactual price = 160*0.8 = 128.\nAnswer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125\nNumber of bags sold = 3000/125 = 24\nAnswer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If the percentage of black marbles pulled out the second time represents their percentage in the bag, how many marbles in total Q does the bag currently hold?
Answer Choices: (A) 40 (B) 200 (C) 380 (D) 400 (E) 3200
Rationale: We know that there are 20 black marbles in the bag and this number represent 1/20 th of the number of all marbles in the bag, thus there are total Q of 20*20=400 marbles.\nAnswer: D.
Correct Answer: D


Question: Janice bikes at 10 miles per hour, while Jennie bikes at 20. How long until they have collectively biked 1 mile?
Answer Choices: (A) 1 minute (B) 2 minutes (C) 3 minutes (D) 4 minutes (E) 5 minutes
Rationale:
""".strip()

In [508]:
batch = tokenizer(EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, return_tensors='pt', return_token_type_ids=False).to(device)
torch.manual_seed(1337)
output_tokens = model.generate(**batch, max_new_tokens=100, do_sample=True, top_p=0.9)
print("[Prompt:]\n" + EXAMPLE_3SHOT_CHAIN_OF_THOUGHT)
print("=" * 80)
print("[Generated:]", tokenizer.decode(output_tokens[0][batch['input_ids'].shape[1]:].cpu()))
#### NOTE: scroll down for the final answer (below the ======= line)

[Prompt:]
Question: The original retail price of an appliance was 60 percent more than its wholesale cost. If the appliance was actually sold for 20 percent less than the original retail price, then it was sold for what percent more than its wholesale cost?
Answer Choices: (A) 20% (B) 28% (C) 36% (D) 40% (E) 42%
Rationale: wholesale cost = 100;
original price = 100*1.6 = 160;
actual price = 160*0.8 = 128.
Answer: B.
Correct Answer: B


Question: A grocer makes a 25% profit on the selling price for each bag of flour it sells. If he sells each bag for $100 and makes $3,000 in profit, how many bags did he sell?
Answer Choices: (A) 12 (B) 16 (C) 24 (D) 30 (E) 40
Rationale: Profit on one bag: 100*1.25= 125
Number of bags sold = 3000/125 = 24
Answer is C.
Correct Answer: C


Question: 20 marbles were pulled out of a bag of only white marbles, painted black, and then put back in. Then, another 20 marbles were pulled out, of which 1 was black, after which they were all returned to the bag. If 

__Task 6 (1 pt)__ write a function that automatically creates chain-of-thought prompts. Follow the instructions from the function docstring.

In [566]:
QUESTION_PREFIX = "Question: "
OPTIONS_PREFIX = "Answer Choices: "
CHAIN_OF_THOUGHT_PREFIX = "Rationale: "
ANSWER_PREFIX = "Correct Answer: "
FEWSHOT_SEPARATOR = "\n\n\n"

import re

def process_option(option_str):
    regex = '[A-Z]\)'

    return re.sub(regex, lambda match: '(' + match.group() +' ' , option_str)

def process_options(options_list):

    return OPTIONS_PREFIX + ' '.join(map(process_option, options_list))

def make_prompt(*, main_question, fewshot_examples):
    """
    Your goal is to produce the same prompt as the EXAMPLE_3SHOT_CHAIN_OF_THOUGHT automatically

    For each few-shot question, make sure to follow the following rules:
    1. Each question begins with QUESTION_PREFIX, after which you should print the question without leading/traiiling spaces (if any)
    2. After the question, provide space-separated options. Each option should be put in double brackets, followed by option text, e.g. "(A) 146%"
    3. Then, provide the answer as a single letter (A-E)
    4. Finally, add trailing newlines from FEWSHOT_SEPARATOR

    Your final prompt should contain all fewshot_examples (in order), separated with FEWSHOT_SEPARATOR, then follow with main_question.
    The main_question should contain the question and options formatted the same way as in FEWSHOT_EXAMPLES.
    After that, you should prompt the model to produce an explanation (rationale) for the answer.

    Please make sure your prompt contains no leading/trailing newlines or spaces, same as in EXAMPLE_3SHOT_CHAIN_OF_THOUGHT
    """

    prompt = ''
    for example in fewshot_examples:
        prompt += QUESTION_PREFIX + example['question'].strip() + '\n'
        prompt += process_options(example['options']) + '\n'
        prompt += CHAIN_OF_THOUGHT_PREFIX + example['rationale'] + '\n'
        prompt += ANSWER_PREFIX + example['correct']
        prompt += FEWSHOT_SEPARATOR

    prompt += QUESTION_PREFIX + main_question['question'].strip() + '\n'
    prompt += process_options(main_question['options']) + '\n'
    prompt += CHAIN_OF_THOUGHT_PREFIX.strip()

    return prompt



generated_fewshot_prompt = make_prompt(main_question=data[150], fewshot_examples=(data[30], data[20], data[5]))
assert generated_fewshot_prompt == EXAMPLE_3SHOT_CHAIN_OF_THOUGHT, "prompts don't match"
assert generated_fewshot_prompt != make_prompt(main_question=data[150], fewshot_examples=())
assert generated_fewshot_prompt.endswith(make_prompt(main_question=data[150], fewshot_examples=()))

print("Well done!")

# Hint: if two prompts do not match, you may find it usefull to use https://www.diffchecker.com or similar to find the difference

Well done!


__Task 7 (1 points):__ Evaluate your prompt.

Please run the model on the entire dataset and measure it's accuracy.
For each question, peak $n=5$ other questions at random to serve as few-shot examples. Make sure not to accidentally sample the main_question among few-shot examples. For scientific evaluation, it is also a good practice to split the data into two parts: one for eval, and another for few-shot examples. However, doing so is optional in this homework.

The tricky part is when to stop generating: if you don't control for this, your model can accidentally generate a whole new question - and promptyly answer it :) To make sure you get the correct answer, stop generating tokens when the model is done explaining it's solution. To circumvent this, you need to __stop generating as soon as the model generates Final Answer: [A-E]__
To do so, you can either generate manually (see low-level generation above) or use [transformers stopping criteria](https://discuss.huggingface.co/t/implimentation-of-stopping-criteria-list/20040/2), whichever you prefer.

If you do everything right, the model should be much better than random. However, please __do not expect miracles__: this is far from the best models, and it will perform much worse than an average human.

In [709]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side = 'left')

@torch.inference_mode()
def generate_batch(prompts, max_new_tokens=64, **kwargs):

    inputs = tokenizer(prompts, return_tensors='pt', padding = True).to(device) 
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, **kwargs)
    return tokenizer.decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)

In [895]:
from sklearn.model_selection import train_test_split
import numpy as np

examples_data, eval_data = train_test_split(data, test_size=0.4)
LETTERS = [chr(ord('A') + i) for i in range(26)]
ANSWERS = [chr(ord('A') + i) for i in range(5)]
stopping_criteria_list = StoppingCriteriaList([StopStringCriteria(tokenizer, [ANSWER_PREFIX + l for l in LETTERS])])

torch.manual_seed(0)
np_gen = np.random.default_rng(seed=0)

def parse_answer(answer):

    match = re.search(fr"{ANSWER_PREFIX}([A-Z])", answer)
    if match is None:
        return None
    return match.group(1)

@torch.inference_mode()
def evaluate_few_shot_prompting(examples_data, 
                                eval_data,
                                batch_size = 1,
                                num_examples = 5, 
                                **generation_kwargs):

    num_samples = 0    # use this to count how many samples you evaluated
    num_responded = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
    num_correct = 0    # how many times did the model's chosen answer (letter) match the correct answer

    bar = tqdm(total = len(eval_data), unit = 'question')
    batch = []

    answers = []

    for i, eval_example in enumerate(eval_data):
    
        index = np_gen.integers(len(examples_data), size = num_examples)
        prompt = make_prompt(main_question=eval_example, fewshot_examples=[examples_data[ind] for ind in index])
        batch.append((prompt, eval_example['correct']))
    
        if len(batch) == batch_size or i == (len(eval_data) - 1):
    
            generated_answers = generate_batch([x[0] for x in batch], 
                                        **generation_kwargs)

            answers.extend(generated_answers)
    
            for j, generated_answer in enumerate(generated_answers):
                parsed_answer = parse_answer(generated_answer)
            
                num_samples += 1
                
                if parsed_answer:
                    num_responded += 1
            
                    if parsed_answer == batch[j][1]:
                        num_correct += 1

            bar.update(len(batch))
            batch = []
            
    
    bar.close()

    return (num_samples, num_responded, num_correct), answers
        
# Optionally, consider inferencing multiple sentences in a batch for faster inference;
# If you choose to batch outputs, make sure the results are the same as with batch=1 (using greedy inference)

In [900]:
gc.collect()
torch.mps.synchronize()
torch.mps.empty_cache()
torch.mps.synchronize()
memory_stats()

Tensor allocated: 8.63 GiB | Driver allocated: 10.98 GiB | Recommended max: 11.84 GiB


In [901]:
(num_samples, num_responded, num_correct), answers = evaluate_few_shot_prompting(
    examples_data, 
    eval_data, 
    batch_size = 4, 
    num_examples = 5, 
    do_sample = True, 
    temperature = 0.8, 
    max_new_tokens = 512,
    top_p = 0.9,
    stopping_criteria = stopping_criteria_list)

  0%|          | 0/102 [00:00<?, ?question/s]

In [902]:
print("Responded %%:", num_responded / num_samples)
print("Accuracy (when responded):", num_correct / num_responded)
print("Accuracy (overall):", num_correct / num_samples)

if num_responded / num_samples < 0.9:
  print("Something is wrong with the evaluation technique (for 5-shot CoT): the model refuses to answer too many questions.")
  print("Make sure you generate enough tokens that the model can produce a correct answer.")
  print("When in doubt, take a look at the full model output. You can often spot errors there.")

Responded %%: 0.9509803921568627
Accuracy (when responded): 0.6494845360824743
Accuracy (overall): 0.6176470588235294


__Task 8 (2 points)__ Experiment time!

Your final quest is to use the testbench you've just written to answer one of the following questions:

### Option 1: How many shots do you need?

How does model accuracy change with the number of fewshot examples?

a. check if the model accuracy changes as you increase/decrease the number of "shots"

b. try to prompt-engineer a model into giving the best rationale __without__ any few-shot examples, i.e. zero-shot

For zero-shot mode, feel free to use wild prompt-engineering or modify the inference procedure.

### Option 2: Is this prompting tecnique reliable?

_Inspired by research by Anton Voronov, Lena Volf and Max Ryabinin._

For this option, you need to check if the model behavior (and hence, accuracy) is robust to perturbations in the input prompt.

a. Does the accuracy degrade if you provide wrong answers to few-shot examples? (make sure to modify rationale if it contains answer in the end)

b. Does it degrade if you replace question/answer prompts with "Q" and "A"? What if you write both on the same line? Change few-shot separators?



### Option 3: Inference Matters

There are many ways to inference the model, not all of them equal.

a. check whether greedy inference or beam search affects model generation quality

b. implement and evaluate sampling with voting (see explanation below).


The voting technique(b) should work as follows: first, you generate k (e.g. 50) "attempts" at an answer using nucleus sampling (or a similar technique).
Then, you count how many of those attempts chose a particular option (A, B, etc) as the final answer. The option that was chosen most frequently has the most "votes", and therefore "wins".

To speed up voting, you may want to generate these attempts in parallel as a batch. That should be very easy to implement: just run `model.generate` on a list with multiple copies of the same prompt.




================================================

__Common rules:__ You will need to test both hypothes (A and B) in the chosen option. You may choose to replace one of them with your own idea - but please ask course staff in advance (via telegram) if you want full points.

Feel free to organize your code and report as you see fit - but please make sure it's readable and the code runs top-to-bottom :)
Write a short informal report about what you tried and, in doing so, what did you found. Minimum of 2 paragraphs; more is ok; creative visualizations are welcome.

You are allowed (but not required) to prompt the model into generating a report for you --- or helping you write one. However, if you do so, make sure that it is still human-readable :)



In [930]:
def print_summary(num_samples, num_responded, num_correct):

    print("Responded %%:", f"{num_responded / num_samples : .4f}")
    print("Accuracy (when responded):", f"{num_correct / num_responded : .4f}")
    print("Accuracy (overall):", f"{num_correct / num_samples : .4f}")
    

In [939]:
gc.collect()
torch.mps.synchronize()
torch.mps.empty_cache()
torch.mps.synchronize()
memory_stats()

Tensor allocated: 8.69 GiB | Driver allocated: 10.98 GiB | Recommended max: 11.84 GiB


In [936]:
### greedy
(num_samples, num_responded, num_correct), answers = evaluate_few_shot_prompting(
    examples_data, 
    eval_data, 
    batch_size = 4, 
    num_examples = 5, 
    do_sample = False, ### look here
    max_new_tokens = 512,
    stopping_criteria = stopping_criteria_list)

print("GREEDY:")
print_summary(num_samples, num_responded, num_correct)
print("="*80 + '\n')

  0%|          | 0/102 [00:00<?, ?question/s]

GREEDY:
Responded %%:  0.8529
Accuracy (when responded):  0.6552
Accuracy (overall):  0.5588



In [938]:
### beam
(num_samples, num_responded, num_correct), answers = evaluate_few_shot_prompting(
    examples_data, 
    eval_data, 
    batch_size = 1,
    num_examples = 5, 
    num_beams = 4, ### look here
    do_sample = True, 
    temperature = 0.8, 
    max_new_tokens = 512,
    top_p = 0.9,
    stopping_criteria = stopping_criteria_list)

print("BEAM:")
print_summary(num_samples, num_responded, num_correct)
print("="*80 + '\n')

  0%|          | 0/102 [00:00<?, ?question/s]

BEAM:
Responded %%:  0.8431
Accuracy (when responded):  0.7674
Accuracy (overall):  0.6471



In [942]:
from collections import defaultdict
import operator

@torch.inference_mode()
def evaluate_few_shot_prompting_voting(examples_data, 
                                eval_data,
                                batch_size = 1, 
                                num_examples = 5, 
                                **generation_kwargs):

    num_samples = 0    # use this to count how many samples you evaluated
    num_responded = 0  # how many times did the model produce Correct Answer: (letter) in it's response. use as a sanity check.
    num_correct = 0    # how many times did the model's chosen answer (letter) match the correct answer

    bar = tqdm(total = len(eval_data), unit = 'question')

    answers = []

    for i, eval_example in enumerate(eval_data):
    
        index = np_gen.integers(len(examples_data), size = num_examples)
        prompt = make_prompt(main_question=eval_example, fewshot_examples=[examples_data[ind] for ind in index])    
        
        generated_answers = generate_batch([prompt]*batch_size, 
                                    **generation_kwargs)

        answers.append(generated_answers)

        num_samples += 1
        candidates = defaultdict(int)

        for generated_answer in generated_answers:
            parsed_answer = parse_answer(generated_answer)
        
            if parsed_answer:
                candidates[parsed_answer] += 1

        if candidates:
            answer, counts = max(candidates.items(), key=operator.itemgetter(1))        
            num_responded += 1
            if answer == eval_example['correct']:
                num_correct += 1

        bar.update(1)
    
    bar.close()

    return (num_samples, num_responded, num_correct), answers

In [943]:
### voting
(num_samples, num_responded, num_correct), answers = evaluate_few_shot_prompting_voting(
    examples_data, 
    eval_data, 
    batch_size = 4, 
    num_examples = 5, 
    do_sample = True, 
    temperature = 0.8, 
    max_new_tokens = 512,
    top_p = 0.9,
    stopping_criteria = stopping_criteria_list)

print("VOTING:")
print_summary(num_samples, num_responded, num_correct)
print("="*80 + '\n')

  0%|          | 0/102 [00:00<?, ?question/s]

VOTING:
Responded %%:  1.0000
Accuracy (when responded):  0.7255
Accuracy (overall):  0.7255



Results:

Here are the evaluation results for Option 3. I have splitted data in training and eval examples in ratio 60/40, and use the same parameters for generation:

```
num_examples = 5 
do_sample = True 
temperature = 0.8
max_new_tokens = 512
top_p = 0.9
```

except for greedy generation. I have also tried generating shorter sequences, but it resulted in lower responded ratio. It seems that at `max_new_tokens = 512` this measure approaches the plateau: we have enough tokens to generate the final answer, but formatting may be slightly different from what we expect (i.e. `the correct answer is X`). For beam search and voting generation I used 4 candidates. Metric are reported in the table:

| Strategy | % responded | accuracy (responded) | accuracy (overall) |
| :---:   | :---: | :---: |  :---: |
| Nucleus | 95.1% | 64.9% | 61.8% |
| Greedy  | 85.3% | 65.5% | 55.9% | 
| Beam    | 84.3% | **76.7%** | 64.7% |
| Voting  | **100.0%** | 72.5% | **72.5%**|

1. Greedy search produces similar to nucleus accuracy when responded, but it produced correct output only in 85% cases and result in bad overall accuracy.
2. Beam search also suffers from incosistent output formatting but produces much higher responded accuracy, which results in similar overall accuracy.
3. Voting strategy has exceptional ability to respond all questions: when one of candidates fails to answer, he is ignored in voting. This strategy produces a little bit lower responded accuracy then beam search, but in the end it leads to superior overall accuracy.

These results suggest that combining beam search and voting could be very effective solution.
Also note that voting can use effective pre-fill, because we only need to compute KV-cache once (however, I haven't done this). 
While some improvements are obvious, the sample size is small and results should be statistically validated.